# Ejercicio de clase: Predicción del precio de vivienda con Decision Tree Regressor

**Dataset**: California Housing Prices — https://www.kaggle.com/datasets/camnugent/california-housing-prices

En el ejemplo de clase (`decision_tree_example.ipynb`) usamos un **Decision Tree Classifier** para predecir
si un paciente sobrevivía o no a una sepsis. Esa era una tarea de **clasificación**: la variable que
queríamos predecir (`hospital_outcome`) solo podía tomar un número limitado de valores (0 o 1, "vive" o
"muere").

En este ejercicio vamos a resolver un problema distinto: **regresión**. Vamos a predecir el **precio
mediano de una vivienda** (`median_house_value`) en un bloque censal de California, a partir de
características como la ubicación, la cantidad de habitaciones o el ingreso medio de sus habitantes.

> **Diferencia clave**: en clasificación el modelo predice una *categoría* (una clase). En regresión el
> modelo predice un *número real* (puede tomar, en principio, cualquier valor dentro de un rango continuo).
> Esto tiene consecuencias importantes: no podemos usar métricas como *precision*, *recall* o *F1*, porque
> esas métricas comparan clases exactas. En su lugar usaremos métricas que midan **qué tan lejos** está la
> predicción del valor real, como el **MAE (Mean Absolute Error)**, que veremos más adelante.

Para este ejercicio usarán `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.

### Antes de empezar

1. Descarguen el dataset desde Kaggle: https://www.kaggle.com/datasets/camnugent/california-housing-prices
2. El archivo se llama `housing.csv`. Colóquenlo dentro de la carpeta `raw/` de este proyecto
   (la misma carpeta `decision_tree/raw/` donde está el dataset de sepsis).
3. Sigan cada sección en orden. Cada sección tiene:
   - Una breve explicación de qué vamos a hacer y por qué.
   - Un **reto**: ustedes deben escribir el código (no está resuelto).
   - **Preguntas de análisis**: respóndanlas en una celda de markdown justo debajo de su código.


### Importar librerías

Igual que en el ejemplo de clase, empezamos importando las librerías que vamos a necesitar. Noten dos
diferencias frente al ejemplo de clasificación:

- Usamos `DecisionTreeRegressor` en lugar de `DecisionTreeClassifier`.
- Usamos `mean_absolute_error` en lugar de `precision_score`, `recall_score`, `f1_score`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import mean_absolute_error


## Paso 1 — Cargar los datos

**Reto**: Carguen el archivo `housing.csv` (carpeta `raw/`) en un DataFrame de pandas llamado `df`, tal
como hicimos en el ejemplo de clase con `pd.read_csv(...)`. Luego muestren las primeras filas del
DataFrame para confirmar que se cargó correctamente.


In [ ]:
# Cargamos el dataset de California Housing.
df = pd.read_csv("raw/housing.csv")
df.head()

In [ ]:
# Mostramos las primeras filas para verificar la lectura del archivo.
df.head()

**Preguntas de análisis**
1. ¿Cuántas filas y cuántas columnas tiene el dataset? (pista: `df.shape`)
2. ¿Qué representa cada fila del dataset? ¿Es una vivienda individual o algo distinto?
3. Observando los nombres de las columnas, ¿cuál creen que es la variable que vamos a predecir (el
   *target*)?


**Respuesta**
- El dataset tiene 20.640 filas y 10 columnas.
- Cada fila representa una zona censal de California, no una vivienda individual.
- La variable a predecir es `median_house_value`, porque es la que queremos estimar en dólares.


## Paso 2 — Identificación inicial de los datos

Antes de tocar cualquier dato, siempre debemos entender con qué estamos trabajando: cuántas columnas hay,
qué tipo de dato tiene cada una, si hay valores nulos, y cuál es el rango de valores de cada variable.

**Reto**: Usando lo que ya conocen de pandas, respondan (con código) estas tres preguntas:
1. ¿Qué tipo de dato (`dtype`) tiene cada columna? ¿Hay alguna columna categórica (texto)?
2. ¿Cuáles son las estadísticas básicas (media, desviación estándar, mínimo, máximo, etc.) de las
   columnas numéricas?
3. ¿Hay valores nulos (faltantes) en el dataset? ¿En qué columna(s)?

Pistas de los métodos que necesitan (ya los usamos, en otra forma, en el ejemplo de clase): `.info()`,
`.describe()`, `.isnull()`.


In [ ]:
# Revisamos los tipos de cada columna para detectar variables categóricas.
df.info()


In [ ]:
# Resumen estadístico de las columnas numéricas.
df.describe()

In [ ]:
# Contamos valores faltantes por columna para detectar problemas de calidad de datos.
df.isnull().sum()

**Preguntas de análisis**
1. ¿Cuál es la única columna categórica (de texto) del dataset? ¿Qué valores puede tomar?
2. ¿Qué columna tiene valores nulos? ¿Cuántas filas están afectadas, aproximadamente qué porcentaje del
   total representa?
3. Miren el `min` y el `max` de `median_house_value`. ¿Les parece un rango razonable para el precio de una
   vivienda? ¿Notan algo raro en el valor máximo? (pista: busquen cuántas filas tienen exactamente ese
   valor máximo).
4. Comparen el rango de `median_income` con el de `total_rooms`. ¿Están en escalas muy distintas? ¿Creen
   que eso sería un problema para un Decision Tree? (piensen en cómo el árbol elige los cortes: ¿necesita
   que las variables estén en la misma escala, como sí lo necesitan otros modelos?)


**Respuesta**
- La única columna categórica es `ocean_proximity`, y toma valores como `<1H OCEAN`, `INLAND`, `NEAR BAY`, `NEAR OCEAN` e `ISLAND`.
- `total_bedrooms` es la columna con valores nulos; hay 207 filas faltantes, aproximadamente 1% del total.
- `median_house_value` tiene un máximo de 500001, y ese valor aparece muchas veces porque la base usa un límite superior para precios más altos.
- `median_income` está en una escala mucho menor que `total_rooms`, pero los árboles de decisión no necesitan que todas las variables estén en la misma escala; comparan cortes en cada variable por separado.


## Paso 3 — Procesamiento de datos

Por ahora, para mantener el ejercicio simple, vamos a trabajar **solo con variables numéricas**. Más
adelante en el curso aprenderemos técnicas para incorporar variables categóricas (como *one-hot encoding*),
pero hoy las vamos a descartar.

**Reto**:
1. Eliminen del DataFrame la(s) columna(s) categórica(s) que identificaron en el paso anterior.
2. Decidan qué hacer con los valores nulos que encontraron (por ejemplo, eliminar esas filas) y
   apliquen esa decisión. Un Decision Tree Regressor de scikit-learn no puede entrenarse si quedan
   valores `NaN` en los datos.
3. Confirmen, con código, que ya no quedan columnas categóricas ni valores nulos.


In [ ]:
# Eliminamos la columna categórica para trabajar solo con variables numéricas.
df_num = df.drop(columns=['ocean_proximity'])

In [ ]:
# Quitamos las filas con valores nulos para evitar errores en el entrenamiento del árbol.
df_num = df_num.dropna()

In [ ]:
# Confirmamos que no quedan columnas no numéricas ni missing values.
df_num.info()
print("Hay nulos:", df_num.isnull().any().any())
print("Columnas no numéricas:", df_num.select_dtypes(exclude=['number']).columns.tolist())

**Preguntas de análisis**
1. ¿Qué información de las viviendas estamos perdiendo al eliminar la columna categórica? ¿Creen que esa
   información podría ser útil para predecir el precio? ¿Por qué?
2. Si eliminaron filas con nulos, ¿cuántas filas quedaron en total? ¿Qué porcentaje del dataset original
   se perdió?
3. ¿Qué otra estrategia (distinta a eliminar las filas) existe para tratar valores nulos? ¿Por qué hoy
   optamos por la más simple?


**Respuesta**
- Al eliminar `ocean_proximity` perdemos la ubicación geográfica de la zona censal, que normalmente sí ayuda a predecir el precio.
- Si se eliminaron 207 filas, quedan 20.433 registros; eso supone aproximadamente 1% del conjunto original.
- Otra opción es imputar los valores faltantes con la media o la mediana; hoy elegimos eliminar filas porque es la opción más simple y directa para un ejercicio inicial.


## Paso 4 — Definir variables predictoras (X) y variable objetivo (y), y dividir los datos

Igual que en el ejemplo de clase, necesitamos separar:
- **X**: las columnas que el modelo usará para predecir (todas menos el precio).
- **y**: la columna que queremos predecir (`median_house_value`).

Y luego dividir ambas en un conjunto de **entrenamiento** y uno de **prueba**, usando
`train_test_split`, tal como hicimos con los datos de sepsis.

**Reto**:
1. Construyan `X` (todas las columnas numéricas excepto `median_house_value`) y `y`
   (`median_house_value`).
2. Usen `train_test_split` para crear `X_train`, `X_test`, `y_train`, `y_test`. Usen un `test_size` de
   0.2 y `random_state=0` para que los resultados sean reproducibles.


In [ ]:
# X contiene las características y y contiene la variable objetivo.
X = df_num.drop(columns=['median_house_value'])
y = df_num['median_house_value']

In [ ]:
# Separamos datos en entrenamiento y prueba con una semilla fija para reproducibilidad.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0
)

**Preguntas de análisis**
1. ¿Cuántas filas quedaron en `X_train` y cuántas en `X_test`?
2. ¿Por qué es importante evaluar el modelo en datos que **no** usó para entrenar (`X_test`, `y_test`)?
   ¿Qué pasaría si evaluáramos únicamente sobre `X_train`?
3. En el ejemplo de clase balanceamos las clases con SMOTE antes de dividir los datos. Aquí no lo hicimos.
   ¿Por qué SMOTE (que genera ejemplos sintéticos de una *clase* minoritaria) no tiene sentido en un
   problema de regresión, donde no hay clases sino un valor continuo?


**Respuesta**
- En `X_train` hay 16.346 filas y en `X_test` 4.087 filas.
- Evaluamos en datos no vistos porque así comprobamos si el modelo generaliza y no solo memoriza el conjunto de entrenamiento.
- SMOTE no aplica a regresión porque no hay clases discretas; allí se predice un valor continuo, no una categoría.


## Paso 5 — Entrenar el Decision Tree Regressor

**Reto**: Entrenen un `DecisionTreeRegressor` (con `random_state=0`) usando `X_train` y `y_train`, de la
misma forma en que entrenaron el `DecisionTreeClassifier` en el ejemplo de clase.


In [ ]:
# Entrenamos el modelo de regresión con árbol de decisión.
reg = DecisionTreeRegressor(random_state=0)
reg.fit(X_train, y_train)

**Preguntas de análisis**
1. En clasificación, cada hoja del árbol predice una clase (por ejemplo, "vive" o "muere"). En regresión,
   ¿qué creen que predice cada hoja del árbol? (pista: piensen en los valores de `y` que caen en esa
   hoja durante el entrenamiento).
2. ¿Qué criterio usa por defecto `DecisionTreeRegressor` para decidir dónde hacer cada corte, en lugar del
   *gini* o *entropy* que se usan en clasificación? (revisen la documentación del parámetro `criterion`).


**Respuesta**
- En cada hoja, el árbol predice el valor medio de `median_house_value` de todas las observaciones que cayeron en esa hoja.
- El criterio por defecto es `squared_error`, que busca reducir la varianza de los valores objetivo dentro de cada partición.


## Paso 6 — Evaluar el modelo con MAE

### ¿Qué es el MAE (Mean Absolute Error)?

El **MAE** es el promedio de la diferencia absoluta entre el valor real y el valor predicho:

$$MAE = \frac{1}{n}\sum_{i=1}^{n} |y_i - \hat{y}_i|$$

A diferencia de precision/recall/F1 (que solo tienen sentido cuando comparamos clases), el MAE funciona
sobre valores numéricos continuos y nos dice, **en promedio, cuánto se equivoca el modelo, en las mismas
unidades que la variable objetivo**. En nuestro caso, como `median_house_value` está en dólares, un
MAE de, por ejemplo, 40000 significa que en promedio el modelo se equivoca en $40,000 dólares al predecir
el precio de una vivienda.

Un MAE más bajo es mejor. Pero un mismo valor de MAE puede ser "bueno" o "malo" dependiendo de la escala
de la variable que estamos prediciendo — por eso siempre hay que compararlo contra algo (por ejemplo,
el precio promedio de las viviendas).

**Reto**:
1. Usen el modelo entrenado para predecir sobre `X_train` y calculen el MAE comparando esas predicciones
   con `y_train`.
2. Hagan lo mismo sobre `X_test` con `y_test`.
3. Comparen ambos valores.


In [ ]:
# Calculamos las predicciones del conjunto de entrenamiento para medir ajuste inicial.
y_train_pred = reg.predict(X_train)
mae_train = mean_absolute_error(y_train, y_train_pred)
print("MAE (train):", mae_train)


In [ ]:
# Repetimos la medición en el conjunto de prueba para ver la generalización.
y_test_pred = reg.predict(X_test)
mae_test = mean_absolute_error(y_test, y_test_pred)
print("MAE (test):", mae_test)


**Preguntas de análisis**
1. ¿El MAE de entrenamiento es mayor, menor o similar al MAE de prueba? ¿Qué les dice eso sobre qué tan
   bien "memorizó" el árbol los datos de entrenamiento?
2. Calculen el precio promedio (`.mean()`) de `median_house_value` en todo el dataset. Comparando ese
   promedio con el MAE de prueba, ¿el error del modelo les parece grande o pequeño en proporción al precio
   típico de una vivienda?
3. En el ejemplo de clase, un árbol sin restricciones (`fully grown`) mostraba señales de sobreajuste
   (*overfitting*) al compararlo con un árbol más simple (`max_depth=3`). Según los MAE de train y test
   que obtuvieron, ¿creen que este árbol también está sobreajustado? ¿Por qué?


**Respuesta**
- Normalmente el MAE de entrenamiento será menor que el de prueba; eso indica que el árbol ajusta mejor los datos vistos que los no vistos.
- El precio promedio de la vivienda es alrededor de $206.855, así que un MAE de prueba de unos pocos decenas de miles es un error relativo moderado.
- Si el MAE de entrenamiento es mucho menor que el de prueba, hay señal de sobreajuste: el árbol está aprendiendo ruido del entrenamiento.


## Reto adicional (opcional) — ¿Se puede mejorar el árbol?

En el ejemplo de clase, limitar la profundidad del árbol (`max_depth=3`) cambió el comportamiento del
modelo. Prueben lo mismo aquí:

1. Entrenen un segundo `DecisionTreeRegressor`, esta vez fijando `max_depth` (prueben con distintos
   valores, por ejemplo 3, 5, 10).
2. Calculen el MAE de train y de test para cada valor de `max_depth`.
3. Grafiquen (opcional) el MAE de train y de test contra `max_depth`, como una curva.

**Preguntas de análisis**
1. ¿Qué le pasa al MAE de entrenamiento a medida que aumentan `max_depth`? ¿Y al de prueba?
2. ¿Existe un valor de `max_depth` donde el MAE de prueba deja de mejorar (o empeora)? ¿Qué relación tiene
   eso con el concepto de *overfitting* que vimos en el ejemplo de clase?
3. Miren `reg.feature_importances_` del árbol entrenado. ¿Cuáles son las 2 o 3 variables más importantes
   para predecir el precio de la vivienda? ¿Tiene sentido con lo que ustedes esperarían intuitivamente?


In [ ]:
# Una versión más simple del árbol ayuda a comparar el efecto de limitar la profundidad.
for max_depth in [3, 5, 10]:
    reg_depth = DecisionTreeRegressor(max_depth=max_depth, random_state=0)
    reg_depth.fit(X_train, y_train)
    train_pred = reg_depth.predict(X_train)
    test_pred = reg_depth.predict(X_test)
    mae_train = mean_absolute_error(y_train, train_pred)
    mae_test = mean_absolute_error(y_test, test_pred)
    print(f"max_depth={max_depth} | MAE train={mae_train:.2f} | MAE test={mae_test:.2f}")

# Importancia de variables: qué características más influyen en la predicción.
print("Variables más importantes:")
for name, importance in sorted(
    zip(X.columns, reg.feature_importances_), key=lambda x: x[1], reverse=True
)[:5]:
    print(f"- {name}: {importance:.4f}")
